# Coastal Flood Step 16: Protection-Shed Mangrove Attribution (Memory Safe)

This notebook attributes **all positive avoided EAD** to Forces of Nature mangrove patches using a coastal protection-shed proxy.

Design goals:
- Run in limited memory.
- Do **not** modify DEM files.
- Attribute 100% of positive avoided EAD (with QA check).

Method (vector protection-shed proxy):
1. Build coastline segments from the Jamaica boundary.
2. Link mangrove patches to nearby coastline segments.
3. Load asset-level avoided EAD and asset geometries.
4. Assign each asset to nearest coastline segment.
5. Distribute each asset's avoided EAD to linked mangroves using capacity weights.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.ops import substring
from shapely.geometry import LineString, MultiLineString

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 200)


In [ ]:
# -----------------------------
# User parameters
# -----------------------------
SCENARIO = 'minimum'  # 'minimum' or 'maximum'

COAST_SEGMENT_LENGTH_M = 2000.0
MANGROVE_TO_COAST_MAX_DIST_M = 1500.0
MANGROVE_CONNECT_BUFFER_M = 500.0

# Capacity weighting uses area and effective width
USE_CAPACITY_WEIGHTING = True

# Keep negative avoided values separate (not included in attribution totals)
ATTRIBUTE_ONLY_POSITIVE_AVOIDED = True

# Paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / f'dphil_paper_3/results_coastal_{SCENARIO}_scenario'
common_data = base_path / 'dphil_common_cross_cutting/common_incoming_data'

mangroves_path = common_data / 'landcover/mangroves_fn/mangroves.shp'
jamaica_boundary_path = common_data / 'boundaries/jamaica.gpkg'
dem_path = common_data / 'DEM/DEM_30m.tif'  # read-only metadata check

damage_estimates_dir = results_path / 'damage_estimates'
asset_ead_csv = damage_estimates_dir / 'coastal_ead_asset_level_usd.csv'

out_dir = damage_estimates_dir / 'mangrove_attribution_protection_shed'
out_dir.mkdir(parents=True, exist_ok=True)

print('Scenario:', SCENARIO)
print('Output folder:', out_dir)


In [ ]:
# DEM metadata check (read-only)
with rasterio.open(dem_path) as ds:
    print('DEM path:', dem_path)
    print('DEM CRS:', ds.crs)
    print('DEM shape:', (ds.height, ds.width))
    print('DEM resolution:', ds.res)
    print('DEM nodata:', ds.nodata)

print('DEM is not modified in this workflow.')


In [ ]:
# Load mangroves and boundary
mangroves = gpd.read_file(mangroves_path)
boundary = gpd.read_file(jamaica_boundary_path)

if mangroves.crs is None:
    raise ValueError('Mangroves CRS is missing.')

if str(mangroves.crs).upper() != 'EPSG:3448':
    mangroves = mangroves.to_crs('EPSG:3448')

if str(boundary.crs).upper() != 'EPSG:3448':
    boundary = boundary.to_crs('EPSG:3448')

mangroves = mangroves.reset_index(drop=True).copy()
mangroves['Mangrove_ID'] = np.arange(1, len(mangroves) + 1)

# Capacity metrics
mangroves['area_m2'] = mangroves.geometry.area
mangroves['perim_m'] = mangroves.geometry.length
mangroves['effective_width_m'] = (2.0 * mangroves['area_m2'] / mangroves['perim_m'].replace(0, np.nan)).fillna(0.0)
mangroves['capacity_weight_raw'] = mangroves['area_m2'] * mangroves['effective_width_m'].clip(lower=1.0)

print('Mangrove patches:', len(mangroves))
print('Total mangrove area (km2):', round(float(mangroves['area_m2'].sum() / 1e6), 2))


In [ ]:
# Build coastline segments

def split_line_into_segments(line, segment_length):
    if line.is_empty or line.length == 0:
        return []
    n = max(1, int(np.ceil(line.length / segment_length)))
    distances = np.linspace(0, line.length, n + 1)
    out = []
    for i in range(n):
        seg = substring(line, float(distances[i]), float(distances[i + 1]))
        if not seg.is_empty and seg.length > 0:
            out.append(seg)
    return out

coast_geom = boundary.geometry.iloc[0].boundary
line_geoms = list(coast_geom.geoms) if isinstance(coast_geom, MultiLineString) else [coast_geom]

segments = []
for ln in line_geoms:
    segments.extend(split_line_into_segments(ln, COAST_SEGMENT_LENGTH_M))

coast_segments = gpd.GeoDataFrame(
    {'segment_id': np.arange(1, len(segments) + 1)},
    geometry=segments,
    crs='EPSG:3448'
)

print('Coast segments:', len(coast_segments))
print('Mean segment length (m):', round(float(coast_segments.length.mean()), 1))


In [ ]:
# Link mangroves to coast segments (intersects with buffered mangroves)
mangrove_buffers = mangroves[['Mangrove_ID', 'capacity_weight_raw', 'geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(MANGROVE_CONNECT_BUFFER_M)

seg_to_patch = gpd.sjoin(
    coast_segments[['segment_id', 'geometry']],
    mangrove_buffers[['Mangrove_ID', 'capacity_weight_raw', 'geometry']],
    how='left',
    predicate='intersects'
)[['segment_id', 'Mangrove_ID', 'capacity_weight_raw']]

# For segments with no intersected mangrove, use nearest mangrove to ensure full assignment
covered_seg_ids = set(seg_to_patch.loc[seg_to_patch['Mangrove_ID'].notna(), 'segment_id'].astype(int).unique())
missing_seg_ids = set(coast_segments['segment_id']) - covered_seg_ids

if missing_seg_ids:
    missing_seg = coast_segments[coast_segments['segment_id'].isin(sorted(missing_seg_ids))].copy()
    nearest_fill = gpd.sjoin_nearest(
        missing_seg[['segment_id', 'geometry']],
        mangroves[['Mangrove_ID', 'capacity_weight_raw', 'geometry']],
        how='left',
        distance_col='dist_to_mangrove_m'
    )[['segment_id', 'Mangrove_ID', 'capacity_weight_raw']]
    seg_to_patch = pd.concat([seg_to_patch, nearest_fill], ignore_index=True)

seg_to_patch = seg_to_patch.dropna(subset=['segment_id', 'Mangrove_ID']).copy()
seg_to_patch['segment_id'] = seg_to_patch['segment_id'].astype(int)
seg_to_patch['Mangrove_ID'] = seg_to_patch['Mangrove_ID'].astype(int)

if USE_CAPACITY_WEIGHTING:
    seg_to_patch['weight_raw'] = seg_to_patch['capacity_weight_raw'].fillna(0).clip(lower=1.0)
else:
    seg_to_patch['weight_raw'] = 1.0

seg_to_patch['weight_sum'] = seg_to_patch.groupby('segment_id')['weight_raw'].transform('sum')
seg_to_patch['segment_patch_weight'] = seg_to_patch['weight_raw'] / seg_to_patch['weight_sum']

print('Segment-to-patch rows:', len(seg_to_patch))
print('Segments covered:', seg_to_patch['segment_id'].nunique(), '/', len(coast_segments))


In [ ]:
# Load avoided EAD table
asset_ead = pd.read_csv(asset_ead_csv)

if ATTRIBUTE_ONLY_POSITIVE_AVOIDED:
    asset_ead_use = asset_ead.loc[asset_ead['Avoided_EAD_USD'] > 0].copy()
else:
    asset_ead_use = asset_ead.copy()

asset_ead_use['Asset_ID'] = asset_ead_use['Asset_ID'].astype(str)

print('Total rows in EAD table:', len(asset_ead))
print('Rows used for attribution:', len(asset_ead_use))
print('Input avoided total (USD):', float(asset_ead_use['Avoided_EAD_USD'].sum()))


In [ ]:
# Build asset geometry points by (Asset, Layer)

def infer_id_column(cols, layer):
    preferred = ['edge_id', 'node_id', 'id', 'osm_id']
    for c in preferred:
        if c in cols:
            return c
    fallback = [c for c in cols if c.endswith('_id')]
    if fallback:
        return fallback[0]
    raise KeyError(f'Could not infer ID column from columns: {cols}')

asset_points_parts = []
combo_counts = []

for (asset_name, layer_name), sub in asset_ead_use.groupby(['Asset', 'Layer'], dropna=False):
    gpkg_path = damage_estimates_dir / f'{asset_name}_{layer_name}_asset_damages_groupedby.gpkg'
    if not gpkg_path.exists():
        print('Missing geometry file, skipped:', gpkg_path.name)
        continue

    gdf = gpd.read_file(gpkg_path)
    id_col = infer_id_column(gdf.columns, layer_name)

    geom_df = gdf[[id_col, 'geometry']].copy()
    geom_df['Asset_ID'] = geom_df[id_col].astype(str)
    geom_df = geom_df.drop(columns=[id_col])
    geom_df = geom_df.dropna(subset=['geometry'])

    # Representative point for any geometry type
    geom_df['geometry'] = geom_df.geometry.representative_point()
    geom_df = geom_df.drop_duplicates(subset=['Asset_ID'])

    merged = sub.merge(geom_df[['Asset_ID', 'geometry']], on='Asset_ID', how='left')
    missing_geom = merged['geometry'].isna().sum()

    merged = merged.dropna(subset=['geometry']).copy()
    part = gpd.GeoDataFrame(merged, geometry='geometry', crs=gdf.crs)
    if str(part.crs).upper() != 'EPSG:3448':
        part = part.to_crs('EPSG:3448')

    asset_points_parts.append(part)
    combo_counts.append({
        'Asset': asset_name,
        'Layer': layer_name,
        'rows_in_ead': len(sub),
        'rows_with_geometry': len(part),
        'rows_missing_geometry': int(missing_geom),
        'source_file': gpkg_path.name
    })

asset_points = gpd.GeoDataFrame(pd.concat(asset_points_parts, ignore_index=True), geometry='geometry', crs='EPSG:3448')
combo_counts_df = pd.DataFrame(combo_counts).sort_values(['Asset', 'Layer'])

display(combo_counts_df)
print('Total asset rows with geometry:', len(asset_points))
print('Total avoided (geometry-backed) USD:', float(asset_points['Avoided_EAD_USD'].sum()))


In [ ]:
# Assign each asset point to nearest coastline segment
asset_with_segment = gpd.sjoin_nearest(
    asset_points,
    coast_segments[['segment_id', 'geometry']],
    how='left',
    distance_col='dist_to_segment_m'
)

missing_segments = asset_with_segment['segment_id'].isna().sum()
print('Rows missing segment assignment:', int(missing_segments))

asset_with_segment['segment_id'] = asset_with_segment['segment_id'].astype(int)
asset_with_segment = asset_with_segment.drop(columns=['index_right'])

display(asset_with_segment[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'segment_id', 'dist_to_segment_m']].head())


In [ ]:
# Distribute avoided EAD from assets -> mangrove patches via segment weights
asset_patch = asset_with_segment.merge(
    seg_to_patch[['segment_id', 'Mangrove_ID', 'segment_patch_weight']],
    on='segment_id',
    how='left'
)

asset_patch['Avoided_EAD_USD_attributed'] = asset_patch['Avoided_EAD_USD'] * asset_patch['segment_patch_weight']

# QA: attribution completeness
input_total = float(asset_points['Avoided_EAD_USD'].sum())
attributed_total = float(asset_patch['Avoided_EAD_USD_attributed'].sum())
residual = input_total - attributed_total

print('Input total avoided USD:', round(input_total, 6))
print('Attributed total avoided USD:', round(attributed_total, 6))
print('Residual USD (should be ~0):', round(residual, 6))


In [ ]:
# Summaries and outputs
patch_totals = (
    asset_patch.groupby('Mangrove_ID', as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Avoided_EAD_USD_attributed': 'Total_Avoided_EAD_USD_attributed'})
)

patch_sector = (
    asset_patch.groupby(['Mangrove_ID', 'Sector'], as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
)

mangrove_map = mangroves.merge(patch_totals, on='Mangrove_ID', how='left')
mangrove_map['Total_Avoided_EAD_USD_attributed'] = mangrove_map['Total_Avoided_EAD_USD_attributed'].fillna(0.0)

# Save
asset_patch_out = out_dir / f'asset_to_mangrove_attribution_protection_shed_{SCENARIO}.csv'
patch_totals_out = out_dir / f'mangrove_attribution_total_protection_shed_{SCENARIO}.csv'
patch_sector_out = out_dir / f'mangrove_attribution_by_sector_protection_shed_{SCENARIO}.csv'
patch_map_out = out_dir / f'mangrove_attribution_total_protection_shed_{SCENARIO}.gpkg'
combo_counts_out = out_dir / f'asset_geometry_join_qc_{SCENARIO}.csv'

asset_patch[[
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
    'Avoided_EAD_USD', 'segment_id', 'dist_to_segment_m',
    'Mangrove_ID', 'segment_patch_weight', 'Avoided_EAD_USD_attributed'
]].to_csv(asset_patch_out, index=False)

patch_totals.to_csv(patch_totals_out, index=False)
patch_sector.to_csv(patch_sector_out, index=False)
combo_counts_df.to_csv(combo_counts_out, index=False)
mangrove_map.to_file(patch_map_out, driver='GPKG')

print('Saved:')
print(' -', asset_patch_out)
print(' -', patch_totals_out)
print(' -', patch_sector_out)
print(' -', combo_counts_out)
print(' -', patch_map_out)


In [ ]:
# Quick diagnostic plot
ax = mangrove_map.plot(
    column='Total_Avoided_EAD_USD_attributed',
    figsize=(8, 8),
    legend=True,
    cmap='YlGnBu',
    linewidth=0.1,
    edgecolor='black'
)
ax.set_title(f'Mangrove attribution (protection-shed proxy) - {SCENARIO}')
ax.set_axis_off()


## Notes
- This is a **protection-shed proxy** (segment connectivity), not a full hydraulic model.
- DEM is read only for metadata in this notebook; no DEM files are changed.
- If needed, a second version can add DEM-based least-cost connectivity while still writing only outputs to `results`.
